# KionTTS Production Training Pipeline (Multi-GPU / Dual T4 / A100)

This notebook trains a **production-grade, single-speaker StyleTTS2 model with continuous 24-tag emotion & delivery style conditioning**.

### Key Architecture Highlights:
* **LibriTTS Pretrained Backbone**: 100% aligned with official 178-token IPA dictionary.
* **Dual-Stage Phased Training**:
  * **Stage 1 (Acoustic & Prosody Adaptation)**: Aligns `KionTagStyleEncoder` to Kion's acoustic style vectors while adapting PL-BERT & prosody predictor.
  * **Stage 2 (HiFi-GAN & GAN Discriminators)**: Fine-tunes the neural vocoder using `MultiResolutionSTFTLoss`, MPD & MSD discriminators with windowed micro-batching (eliminates OOM).
* **Hugging Face Hub Sync**: Automatic checkpoint upload, resumption, and rolling pruning for Kaggle/Colab 12-hour sessions.

## 1. System Setup & Dependencies

In [ ]:
import os, sys, subprocess, torch

print(f"[*] PyTorch Version : {torch.__version__}")
print(f"[*] CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"    [GPU {i}] {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GB")

# Install system audio libraries
!apt-get update -qq && apt-get install -y -qq espeak-ng libsndfile1 ffmpeg

# Install deep learning and audio packages
!pip install -q phonemizer librosa soundfile torchaudio transformers munch pyworld praat-parselmouth huggingface_hub accelerate tensorboard

## 2. Clone Repository & Setup Working Directory

In [ ]:
import os, sys

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else os.getcwd()
REPO_DIR = os.path.join(WORKING_DIR, "KionTTS") if os.path.exists("/kaggle") else WORKING_DIR

if os.path.exists("/kaggle") and not os.path.exists(REPO_DIR):
    # Check if attached via Kaggle Input
    cand = "/kaggle/input/kiontts"
    if os.path.exists(cand):
        !cp -r /kaggle/input/kiontts /kaggle/working/KionTTS
    else:
        !git clone https://github.com/C1ph3r404/KionTTS.git /kaggle/working/KionTTS

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "StyleTTS2"))
os.chdir(REPO_DIR)
print(f"[✓] Active Directory: {os.getcwd()}")

## 3. Download StyleTTS2 Pretrained Utilities

In [ ]:
import os, urllib.request

def ensure_pretrained():
    base_dir = os.path.join(REPO_DIR, "StyleTTS2")
    asr_dir = os.path.join(base_dir, "Utils/ASR")
    jdc_dir = os.path.join(base_dir, "Utils/JDC")
    plb_dir = os.path.join(base_dir, "Utils/PLBERT")
    libri_dir = os.path.join(REPO_DIR, "Models/LibriTTS")

    os.makedirs(asr_dir, exist_ok=True)
    os.makedirs(jdc_dir, exist_ok=True)
    os.makedirs(plb_dir, exist_ok=True)
    os.makedirs(libri_dir, exist_ok=True)

    downloads = [
        ("ASR epoch_00080.pth", "https://github.com/yl4579/StyleTTS2/raw/main/Utils/ASR/epoch_00080.pth", os.path.join(asr_dir, "epoch_00080.pth")),
        ("ASR config.yml", "https://github.com/yl4579/StyleTTS2/raw/main/Utils/ASR/config.yml", os.path.join(asr_dir, "config.yml")),
        ("JDC bst.t7", "https://github.com/yl4579/StyleTTS2/raw/main/Utils/JDC/bst.t7", os.path.join(jdc_dir, "bst.t7")),
        ("PLBERT step_1000.t7", "https://github.com/yl4579/StyleTTS2/raw/main/Utils/PLBERT/step_1000.t7", os.path.join(plb_dir, "step_1000.t7")),
        ("PLBERT config.yml", "https://github.com/yl4579/StyleTTS2/raw/main/Utils/PLBERT/config.yml", os.path.join(plb_dir, "config.yml")),
    ]

    for name, url, dest in downloads:
        if not os.path.exists(dest) or os.path.getsize(dest) == 0:
            print(f"[*] Downloading {name}...")
            urllib.request.urlretrieve(url, dest)
            print(f"[✓] Downloaded {name}")

    # LibriTTS 2nd stage checkpoint
    libri_pth = os.path.join(libri_dir, "epochs_2nd_00020.pth")
    if not os.path.exists(libri_pth):
        print("[*] Downloading Pretrained LibriTTS 2nd stage checkpoint (epochs_2nd_00020.pth)...")
        from huggingface_hub import hf_hub_download
        downloaded = hf_hub_download(repo_id="yl4579/StyleTTS2-LibriTTS", filename="Models/LibriTTS/epochs_2nd_00020.pth", local_dir=REPO_DIR)
        print(f"[✓] LibriTTS Checkpoint ready: {downloaded}")

ensure_pretrained()

## 4. Hugging Face Hub Authentication & Checkpoint Sync Manager

In [ ]:
import os
from kion_core.checkpoint_manager import KionCheckpointManager, get_hf_token

# 1. Retrieve Hugging Face token from Kaggle Secrets
hf_token = get_hf_token()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("[✓] Hugging Face token authenticated via Kaggle Secrets!")
else:
    print("[!] Notice: HF_TOKEN not detected in Kaggle Secrets. Please add 'HF_TOKEN' under Kaggle Add-ons -> Secrets to enable remote sync.")

# 2. Checkpoint Manager initialized with nate0001/KionTTS
HF_REPO_ID = os.environ.get("HF_REPO_ID", "nate0001/KionTTS")
os.environ["HF_REPO_ID"] = HF_REPO_ID
CHECKPOINT_DIR = os.path.join(REPO_DIR, "checkpoints")

hf_manager = KionCheckpointManager(
    checkpoint_dir=CHECKPOINT_DIR,
    repo_id=HF_REPO_ID,
    hf_token=hf_token,
)


## 5. Dataset Extraction & Manifest Generation
Unpacks `DataSet/data/kion_dataset.tar`, extracts 24kHz mono audio, and generates manifests with the 24 emotion & delivery style tags.

In [ ]:
import os, sys, glob

# Auto-discover dataset source (Kaggle auto-extracted directory, sub-zips, or local tar)
data_candidates = [
    os.path.join(REPO_DIR, "DataSet/data/kion_dataset.tar"),
    os.path.join(REPO_DIR, "DataSet/data"),
    os.path.join(REPO_DIR, "DataSet"),
    "/kaggle/input/kion-dataset",
    "/kaggle/input/kiontts",
    "/kaggle/input/kiontts-dataset",
    "/kaggle/input/kiontts/DataSet/data/kion_dataset.tar",
    "/kaggle/input/kion-dataset/kion_dataset.tar",
]

dataset_source = None
for cand in data_candidates:
    if os.path.exists(cand):
        if os.path.isdir(cand) and len(os.listdir(cand)) > 0:
            dataset_source = cand
            break
        elif os.path.isfile(cand):
            dataset_source = cand
            break

# If not found in standard paths, scan /kaggle/input dynamically
if not dataset_source and os.path.exists("/kaggle/input"):
    print("[*] Scanning /kaggle/input for dataset...")
    for root, dirs, files in os.walk("/kaggle/input"):
        if "metadata.json" in files or any(f.endswith(".zip") for f in files) or "kion_dataset.tar" in files:
            dataset_source = root
            break

if not dataset_source:
    raise FileNotFoundError(
        "Could not find KionTTS dataset! Please ensure 'kion-dataset' or 'kiontts' is added to Kaggle input."
    )

print(f"[✓] Detected dataset source: {dataset_source}")

manifest_train = os.path.join(REPO_DIR, "DataSet/train_manifest.json")
if not os.path.exists(manifest_train) or os.path.getsize(manifest_train) == 0:
    print(f"[*] Preparing dataset from: {dataset_source}")
    !python3 scripts/prepare_dataset.py --source "{dataset_source}" --output_dir DataSet
else:
    print(f"[✓] Manifest already exists: {manifest_train}")


## 6. Stage 1 Training: Acoustic Foundation & Tag Style Alignment
Adapts PL-BERT, Duration Predictor, F0/Energy Predictor, and grounds `KionTagStyleEncoder` to Kion's acoustic style vectors.

In [ ]:
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
launch_cmd = "accelerate launch --multi_gpu" if num_gpus > 1 else "python3"

stage1_cmd = f"""{launch_cmd} scripts/train_stage1.py \
    --manifest DataSet/train_manifest.json \
    --val_manifest DataSet/val_manifest.json \
    --data_root DataSet \
    --checkpoint_dir checkpoints \
    --epochs 40 \
    --batch_size 4 \
    --accum_steps 2 \
    --save_step_freq 500 \
    --hf_repo {HF_REPO_ID} \
    --lr 0.0001
"""
!{stage1_cmd}


## 7. Stage 2 Training: Full-Stack HiFi-GAN Vocoder & GAN Discriminators
Joint fine-tuning with Multi-Resolution STFT reconstruction loss and MPD/MSD GAN guidance.

In [ ]:
stage2_cmd = f"""{launch_cmd} scripts/train_stage2.py \
    --manifest DataSet/train_manifest.json \
    --val_manifest DataSet/val_manifest.json \
    --data_root DataSet \
    --checkpoint_dir checkpoints \
    --epochs 50 \
    --batch_size 4 \
    --accum_steps 2 \
    --save_step_freq 500 \
    --dec_window 48 \
    --hf_repo {HF_REPO_ID} \
    --lr 0.0001
"""
!{stage2_cmd}


## 8. Interactive Audio Inference & Emotional Tag Testing
Synthesizes custom speech prompts with direct inline emotional and delivery style tags.

In [ ]:
import IPython.display as ipd
import soundfile as sf
from kion_core.synthesizer import KionSynthesizer
from kion_core.tag_style_encoder import KionTagStyleEncoder
from scripts.train_stage1 import load_styletts2_backbone

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load backbone & tag encoder
model, _ = load_styletts2_backbone("StyleTTS2/Configs/config_ft.yml", "Models/LibriTTS/epochs_2nd_00020.pth", device=device)
tag_encoder = KionTagStyleEncoder(num_tags=24, emb_dim=64, style_dim=256)

# Find latest checkpoint
best_ckpt = hf_manager.find_latest_checkpoint(stage="stage2") or hf_manager.find_latest_checkpoint(stage="stage1")
if best_ckpt and os.path.exists(best_ckpt):
    print(f"[*] Loading trained checkpoint: {best_ckpt}")
    hf_manager.load_checkpoint(best_ckpt, models={"tag_encoder": tag_encoder, "predictor": model["predictor"], "decoder": model["decoder"]})

synthesizer = KionSynthesizer(model=model, tag_encoder=tag_encoder, device=device)

# Test prompts across different emotions
test_prompts = [
    '[happy=0.9, excited=0.8] We did it! The complete speech pipeline has converged with studio grade quality!',
    '[sarcasm=0.9] Oh, brilliant. Another batch of GPU tensor operations to keep me entertained.',
    '[soothing=0.85, calm=0.7] Take a deep breath. Every phoneme is aligning perfectly.',
    '[angry=0.85] I told you three times already, do not interrupt the training step!',
]

for i, p in enumerate(test_prompts):
    print(f"\nGenerating: {p}")
    wav = synthesizer.synthesize(prompt=p)
    out_path = f"kion_test_sample_{i+1}.wav"
    sf.write(out_path, wav, 24000)
    ipd.display(ipd.Audio(out_path, rate=24000))

## 9. Final Model Bundle Export & Release

In [ ]:
from kion_core.export import export_kion_model

release_dir = "kiontts_production_release"
export_kion_model(
    model=model,
    tag_encoder=tag_encoder,
    output_dir=release_dir,
)
print(f"[✓] Model successfully bundled for production deployment at: {release_dir}")